# CS781 Phase 4 — Experiment 4 (v3 — reviewed)
## Model1 + BitFit + Hierarchical Long-Document CJPE

**Method:** strict BitFit on the Phase-1 Model1 Prefix Encoder / hybrid LM.

- Every original Phase-1 backbone parameter is frozen **except existing bias parameters**.
- The new hierarchical CJPE head is trainable.
- Long documents use four 384-BPE-token views: **head / middle / evidence / tail**.
- Model selection uses only the leakage-clean 994-case DEV set.
- Final refit may use TRAIN + clean DEV after hyperparameters / training duration are fixed.
- TEST labels or overlapping DEV labels are never used.
- Produces at most three high-value Codabench ZIP variants.

## Build notes

This notebook is based on the exact Phase-1 Model1 architecture from `phase1model1_encoder+hybrid(2).ipynb`:

- vocab size 24,576
- `d_model=768`
- 14 Prefix-Encoder blocks
- 12 attention heads
- FFN 3,072
- sinusoidal positional encoding
- fused `qkv` + `out_proj`
- LayerNorm + GELU FFN
- tied token/output embedding
- `max_positions=2048`

BitFit is implemented strictly: only pre-existing backbone parameters whose names end in `.bias` are unfrozen. LayerNorm **weights** remain frozen.

### Reviewed v3

**Unchanged:** architecture, strict-BitFit rule, head/middle/evidence/tail views, every hyperparameter, loss, LR schedule, early stopping, threshold search, refit recipe, calibration and the three submission ZIPs.

**Refactored / hardened**
- One shared training engine (`AccumulatingTrainer`) replaces the two copy-pasted training loops (Cells 9 and 11). The update rule — AMP, finite-loss guard, overflow-aware LR stepping — is identical.
- Threshold search is vectorised (same macro-F1 values as `sklearn`, ~100× fewer calls).
- **Bug fix:** on a re-run, Cell 2 could select this notebook's own `best_bitfit_cjpe.pt` as the *Phase-1* checkpoint (path contains `model1`, name contains `best`). Output directory is now excluded from the search.
- Parameter-count check is now an exact analytic formula instead of a 110–125 M range.
- `torch.load(weights_only=True)` first, with a logged fallback for trusted legacy files.
- The padding-mask check now compares padded vs un-padded hidden states instead of only checking for finite values.
- Evaluator-vs-training F1 consistency check; atomic JSON/checkpoint writes; config snapshot + checkpoint SHA-256 in the manifest.
- Removed unused imports, a variable-name collision (`bad`), and a global mutable counter (`SELECT_STATS`); duplicated chunk-window code merged; docstrings and type hints added; config is validated; `HF_TOKEN` also read from the environment.

**Additional review hardening in v3**
- Keeps all useful v2 refactors; the BitFit method and hyperparameters are unchanged.
- Adds a hard TRAIN↔CLEAN-DEV disjointness guard.
- Records SHA-256 hashes for the exact train/dev/test files, useful because the hosted DEV snapshot changed during this phase.
- Makes output-directory exclusion compatible with Python versions that do not provide `Path.is_relative_to`.
- Makes the gradient-update path explicitly skip non-finite gradient norms even if AMP is disabled.
- Strengthens the padding-mask diagnostic with a catastrophic-mismatch assertion.


In [1]:
# CELL 1 — imports / configuration
import gc, hashlib, json, math, os, random, re, subprocess, sys, tempfile, time, zipfile
from collections import Counter
from pathlib import Path
from typing import Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import hf_hub_download, login
from sklearn.metrics import f1_score
from torch.utils.data import ConcatDataset, DataLoader, Dataset

# ───────────────────────── reproducibility / small utilities ─────────────────────────
SEED = 1234
NOTEBOOK_VERSION = "v3_REVIEWED"

def seed_everything(seed: int = SEED) -> None:
    """Seed python, numpy and torch (CPU + all CUDA devices)."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

def save_json(obj, path, **kw) -> None:
    """Atomic JSON write (tmp file + rename) so a crash never leaves a half-written file."""
    path = Path(path); tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, **kw)
    os.replace(tmp, path)

def read_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

# ───────────────────────── device / precision ─────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

# ───────────────────────── Hugging Face access ─────────────────────────
def get_hf_token() -> str:
    """Kaggle Secret first, then the HF_TOKEN environment variable."""
    try:
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("HF_TOKEN")
        if tok:
            return tok
    except Exception:
        pass
    return os.environ.get("HF_TOKEN", "")

HF_TOKEN = get_hf_token()
assert HF_TOKEN, "HF_TOKEN missing (add it to Kaggle Secrets or set the HF_TOKEN env var)"
login(token=HF_TOKEN)

HF_REPO = "Exploration-Lab/CS-781-Capstone"
ROOT = Path(os.environ.get("PHASE4_ROOT", "/kaggle/working/phase4_exp4_model1_bitfit"))
ROOT.mkdir(parents=True, exist_ok=True)
SEARCH_DIRS = ["/kaggle/input", "/kaggle/working"]

# Optional explicit paths. Usually leave None and let Cell 2 auto-detect.
MODEL1_CKPT_OVERRIDE = None
TOKENIZER_OVERRIDE = None
EXPECTED_TOKENIZER_SHA256 = None

# Dataset audit values (Phase-4 snapshot)
EXPECTED_TRAIN = 32305
EXPECTED_CLEAN_DEV = 994
EXPECTED_TEST = 1517

# ───────────────────────── exact Phase-1 Model1 architecture ─────────────────────────
VOCAB_SIZE = 24576
D_MODEL = 768
N_LAYERS = 14
N_HEADS = 12
FFN_DIM = 3072
DROPOUT = 0.0
MAX_POSITIONS = 2048

# ───────────────────────── long-document representation ─────────────────────────
N_CHUNKS = 4
CHUNK_LEN = 384
CHUNK_NAMES = ["head", "middle", "evidence", "tail"]
CASE_MODE = "auto"          # "auto" | "lower" | anything else = keep case
COLLAPSE_WS = True
EVIDENCE_CUES = [
    "appeal is allowed", "appeal allowed",
    "appeal is dismissed", "appeal dismissed",
    "petition is allowed", "petition allowed",
    "petition is dismissed", "petition dismissed",
    "we allow", "we dismiss",
    "is hereby allowed", "is hereby dismissed",
    "set aside", "acquitted", "convicted",
    "in the result", "accordingly",
]

# ───────────────────────── strict BitFit + head ─────────────────────────
BITFIT_LR = 8.0e-4
CHUNK_PROJ_DIM = 384
HEAD_DROPOUT = 0.15
HEAD_LR = 5.0e-4

# ───────────────────────── optimisation ─────────────────────────
NUM_EPOCHS = 2
EVAL_PER_EPOCH = 3
PATIENCE_EVALS = 4
MICRO_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
EVAL_BATCH_SIZE = 6
WARMUP_RATIO = 0.06
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
LABEL_SMOOTHING = 0.02
USE_AMP = True
AMP_INIT_SCALE = 4096.0
NUM_WORKERS = 2

THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.005), 3)
THRESH_SMOOTH = 9
RUN_REFIT = True
REFIT_MATCH_ORIGINAL_SCHEDULE = True

AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")

def autocast():
    """fp16 autocast context (no-op on CPU or when USE_AMP is False)."""
    return torch.amp.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED)

CONFIG_KEYS = [
    "SEED", "VOCAB_SIZE", "D_MODEL", "N_LAYERS", "N_HEADS", "FFN_DIM", "DROPOUT", "MAX_POSITIONS",
    "N_CHUNKS", "CHUNK_LEN", "CASE_MODE", "COLLAPSE_WS", "BITFIT_LR", "CHUNK_PROJ_DIM",
    "HEAD_DROPOUT", "HEAD_LR", "NUM_EPOCHS", "EVAL_PER_EPOCH", "PATIENCE_EVALS",
    "MICRO_BATCH_SIZE", "GRAD_ACCUM_STEPS", "EVAL_BATCH_SIZE", "WARMUP_RATIO", "WEIGHT_DECAY",
    "GRAD_CLIP", "LABEL_SMOOTHING", "USE_AMP", "AMP_INIT_SCALE", "THRESH_SMOOTH", "RUN_REFIT",
    "REFIT_MATCH_ORIGINAL_SCHEDULE",
]

def config_snapshot() -> dict:
    """Every hyper-parameter above as a JSON-friendly dict (stored in the manifest)."""
    snap = {k: globals()[k] for k in CONFIG_KEYS}
    snap["THRESHOLD_GRID"] = {"min": float(THRESHOLDS.min()), "max": float(THRESHOLDS.max()),
                              "n": int(THRESHOLDS.size)}
    return snap

def validate_config() -> None:
    assert N_CHUNKS == len(CHUNK_NAMES), "N_CHUNKS must equal len(CHUNK_NAMES)"
    assert CHUNK_LEN <= MAX_POSITIONS, "CHUNK_LEN exceeds the model's max_positions"
    assert D_MODEL % N_HEADS == 0, "D_MODEL must be divisible by N_HEADS"
    assert 0.0 < WARMUP_RATIO < 1.0
    assert MICRO_BATCH_SIZE >= 1 and GRAD_ACCUM_STEPS >= 1 and EVAL_PER_EPOCH >= 1 and NUM_EPOCHS >= 1
    assert 0.0 < float(THRESHOLDS.min()) < float(THRESHOLDS.max()) < 1.0

validate_config()
print("chunks:", N_CHUNKS, "x", CHUNK_LEN)
print("effective document batch:", MICRO_BATCH_SIZE * GRAD_ACCUM_STEPS)
print("BitFit backbone LR:", BITFIT_LR, "| head LR:", HEAD_LR)

Device: cuda
GPU: Tesla T4
chunks: 4 x 384
effective document batch: 16
BitFit backbone LR: 0.0008 | head LR: 0.0005


In [2]:
# CELL 2 — locate the Model1 checkpoint / tokenizer robustly
_SHA_CACHE: dict = {}

def sha256_file(path: str) -> str:
    """SHA-256 of a file (memoised, streamed in 1 MiB blocks)."""
    path = str(path)
    if path not in _SHA_CACHE:
        h = hashlib.sha256()
        with open(path, "rb") as f:
            for blk in iter(lambda: f.read(1 << 20), b""):
                h.update(blk)
        _SHA_CACHE[path] = h.hexdigest()
    return _SHA_CACHE[path]

def list_files(search_dirs=SEARCH_DIRS, exclude: Path = ROOT) -> list:
    """All files under `search_dirs`, skipping this notebook's own output directory."""
    exclude = Path(exclude).resolve(); out = []
    for base in search_dirs:
        if not os.path.isdir(base):
            continue
        for root, _, names in os.walk(base):
            root_resolved = Path(root).resolve()
            try:
                inside_output = root_resolved.is_relative_to(exclude)
            except AttributeError:
                # Python <3.9 compatibility.
                try:
                    root_resolved.relative_to(exclude)
                    inside_output = True
                except ValueError:
                    inside_output = False
            if inside_output:
                continue                      # never mistake our own outputs for Phase-1 inputs
            out.extend(os.path.join(root, n) for n in names)
    return out

PHASE1_HINTS = ("model1", "encoder", "hybrid", "phase1", "enc_full")

def is_phase1_checkpoint(path: str) -> bool:
    low = path.lower(); base = os.path.basename(low)
    return low.endswith(".pt") and (
        "enc_full_best" in base
        or ("model1" in low and "best" in base)
        or ("encoder" in low and "best" in base)
    )

def find_checkpoint(files: list):
    cands = [p for p in files if is_phase1_checkpoint(p)]
    assert cands, "Attach Model1 enc_full_best*.pt or set MODEL1_CKPT_OVERRIDE."
    best = sorted(cands, key=lambda p: (sum(w in p.lower() for w in PHASE1_HINTS), p), reverse=True)[0]
    return best, cands

def find_tokenizer(files: list, ckpt: str):
    # Kaggle may rename tokenizer.json -> tokenizer (1).json, tokenizer (2).json, ...
    cands = [p for p in files
             if os.path.basename(p).lower().startswith("tokenizer") and p.lower().endswith(".json")]
    assert cands, "Attach the Model1 tokenizer JSON or set TOKENIZER_OVERRIDE."
    ckpt_dir = str(Path(ckpt).parent)
    def score(p):
        same_dir = int(str(Path(p).parent) == ckpt_dir)
        common = len(os.path.commonpath([os.path.abspath(ckpt), os.path.abspath(p)]))
        hints = sum(w in p.lower() for w in ("model1", "encoder", "hybrid", "phase1", "enc"))
        return (same_dir, common, hints, p)
    return sorted(cands, key=score, reverse=True)[0], cands

_files = list_files()

if MODEL1_CKPT_OVERRIDE:
    PHASE1_CKPT = str(MODEL1_CKPT_OVERRIDE)
else:
    PHASE1_CKPT, _ckpt_cands = find_checkpoint(_files)
    print("checkpoint candidates:")
    for p in _ckpt_cands: print("  -", p)

_tok_cands = []
if TOKENIZER_OVERRIDE:
    TOKENIZER_FILE = str(TOKENIZER_OVERRIDE)
else:
    TOKENIZER_FILE, _tok_cands = find_tokenizer(_files, PHASE1_CKPT)
    print("tokenizer candidates:")
    for p in _tok_cands: print("  -", p)

assert os.path.isfile(PHASE1_CKPT), PHASE1_CKPT
assert os.path.isfile(TOKENIZER_FILE), TOKENIZER_FILE
print("chosen checkpoint:", PHASE1_CKPT)
print("chosen tokenizer :", TOKENIZER_FILE)
print("tokenizer SHA256 :", sha256_file(TOKENIZER_FILE))

if len(_tok_cands) > 1:
    _hashes = {p: sha256_file(p) for p in _tok_cands}
    for p, h in _hashes.items(): print("  tokenizer", h[:12], p)
    if len(set(_hashes.values())) > 1:
        print("⚠️ Multiple DIFFERENT tokenizer files are attached. "
              "Cell 7 has a Phase-1 LM sanity check; if it fails, set TOKENIZER_OVERRIDE explicitly.")

if EXPECTED_TOKENIZER_SHA256:
    assert sha256_file(TOKENIZER_FILE) == EXPECTED_TOKENIZER_SHA256, "Tokenizer SHA mismatch"

checkpoint candidates:
  - /kaggle/input/datasets/aaryant23/model1-ckpts/enc_full_best (9).pt
tokenizer candidates:
  - /kaggle/input/datasets/aaryant23/model1-ckpts/tokenizer (1).json
chosen checkpoint: /kaggle/input/datasets/aaryant23/model1-ckpts/enc_full_best (9).pt
chosen tokenizer : /kaggle/input/datasets/aaryant23/model1-ckpts/tokenizer (1).json
tokenizer SHA256 : 9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d


In [3]:
# CELL 3 — exact LegalBPE loader (same byte-level format as Phase-1 Model1)
BPE_SPECIALS = ["<pad>", "<unk>", "<s>", "</s>"]
_UNIT_RE = re.compile(r"\S+|\s+")           # compiled once (was re-compiled on every encode call)

class LegalBPE:
    """Byte-level BPE tokenizer; loads the JSON produced during Phase 1 (inference only)."""

    def __init__(self, vocab_size: int = 5000):
        self.vocab_size = int(vocab_size); self.merges = []
        self.itos = list(BPE_SPECIALS); self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"; self.symbol_bytes[s] = bytes([b]); self.stoi[s] = len(self.itos); self.itos.append(s)
        self._reset_encode_cache()

    @property
    def pad_token_id(self): return self.stoi["<pad>"]
    @property
    def unk_token_id(self): return self.stoi["<unk>"]
    @property
    def bos_token_id(self): return self.stoi["<s>"]
    @property
    def eos_token_id(self): return self.stoi["</s>"]
    def __len__(self): return len(self.itos)

    @staticmethod
    def word_to_symbols(word: str) -> list:
        return [f"{b:02x}" for b in word.encode("utf-8")]

    def _reset_encode_cache(self) -> None:
        self._ranks = {(l, r): i for i, (l, r) in enumerate(self.merges)}
        self._word_cache = {}; self._cache_nmerges = len(self.merges)

    def encode_word(self, word: str) -> list:
        """Greedy lowest-rank-first BPE merge of one whitespace-delimited unit."""
        if self._cache_nmerges != len(self.merges):
            self._reset_encode_cache()
        hit = self._word_cache.get(word)
        if hit is not None:
            return list(hit)
        symbols = self.word_to_symbols(word)
        while len(symbols) > 1:
            best = None; rank = None
            for j in range(len(symbols) - 1):
                rr = self._ranks.get((symbols[j], symbols[j + 1]))
                if rr is not None and (rank is None or rr < rank):
                    best = (symbols[j], symbols[j + 1]); rank = rr
            if best is None:
                break
            l, r = best; merged = []; j = 0
            while j < len(symbols):
                if j + 1 < len(symbols) and symbols[j] == l and symbols[j + 1] == r:
                    merged.append(l + r); j += 2
                else:
                    merged.append(symbols[j]); j += 1
            symbols = merged
        if len(self._word_cache) >= 250000:
            self._word_cache.clear()
        self._word_cache[word] = tuple(symbols)
        return symbols

    def encode(self, text) -> list:
        out = []; unk = self.unk_token_id
        for unit in _UNIT_RE.findall(str(text)):
            for s in self.encode_word(unit):
                out.append(self.stoi.get(s, unk))
        return out

    def decode(self, ids) -> str:
        buf = bytearray()
        for idx in ids:
            tok = self.itos[int(idx)]
            if tok not in BPE_SPECIALS:
                buf.extend(self.symbol_bytes[tok])
        return bytes(buf).decode("utf-8", errors="replace")

    @classmethod
    def load_json(cls, path) -> "LegalBPE":
        obj = read_json(path)
        tok = cls(obj["vocab_size"])
        tok.merges = [list(x) for x in obj["merges"]]
        tok.itos = list(obj["itos"]); tok.stoi = {t: i for i, t in enumerate(tok.itos)}
        tok.vocab_size = len(tok.itos); tok.symbol_bytes = {}
        for b in range(256):
            s = f"{b:02x}"; tok.symbol_bytes[s] = bytes([b])
        for l, r in tok.merges:
            tok.symbol_bytes[l + r] = tok.symbol_bytes[l] + tok.symbol_bytes[r]
        tok._reset_encode_cache(); return tok

tokenizer = LegalBPE.load_json(TOKENIZER_FILE)
assert len(tokenizer) == VOCAB_SIZE, (len(tokenizer), VOCAB_SIZE)
_probe = "IN THE SUPREME COURT OF INDIA\nSection 14: ₹100 — legal test."
assert tokenizer.decode(tokenizer.encode(_probe)) == _probe, "tokenizer round-trip failed"
print("Tokenizer PASS:", len(tokenizer))

Tokenizer PASS: 24576


In [4]:
# CELL 4 — download Phase-4 data + build leakage-safe splits (+ text normalisation)
DATA_FILES = {
    "train": "Phase_4_CJPE/Train/train.json",
    "dev":   "Phase_4_CJPE/Train/dev.json",
    "test":  "Phase_4_CJPE/Test/test.json",
}

def dedup_labeled(rows: list, name: str) -> list:
    """Collapse duplicate IDs. Same ID + same label -> keep the longer text.
    Same ID + conflicting labels -> drop the ID entirely."""
    seen = {}; conflicts = set(); text_diff = 0
    for r in rows:
        cid = str(r["id"]); y = int(r["label"]); txt = str(r["text"])
        assert y in (0, 1), f"{name}: label {y!r} for {cid}"
        if cid in seen:
            if seen[cid]["label"] != y:
                conflicts.add(cid)
            elif seen[cid]["text"] != txt:
                text_diff += 1
                if len(txt) > len(seen[cid]["text"]):
                    seen[cid]["text"] = txt
        else:
            seen[cid] = {"id": cid, "text": txt, "label": y}
    for cid in conflicts:
        seen.pop(cid, None)
    print(f"{name}: rows={len(rows)} unique={len(seen) + len(conflicts)} | "
          f"same-ID different-text={text_diff} | label-conflict IDs dropped={len(conflicts)}")
    return list(seen.values())

def upper_ratio(text: str, limit: int = 20000) -> float:
    text = text[:limit]
    letters = sum(c.isalpha() for c in text)
    return sum(c.isupper() for c in text) / max(letters, 1)

def sample_upper_ratios(rows: list, rng: random.Random, k: int = 1500) -> np.ndarray:
    idx = rng.sample(range(len(rows)), min(k, len(rows)))
    return np.array([upper_ratio(rows[i]["text"]) for i in idx])

def normalize_text(t: str, lowercase: bool, collapse_ws: bool) -> str:
    if lowercase:
        t = t.lower()
    if collapse_ws:
        t = re.sub(r"\s+", " ", t).strip()
    return t

paths = {k: hf_hub_download(repo_id=HF_REPO, repo_type="dataset", filename=v, token=HF_TOKEN)
         for k, v in DATA_FILES.items()}
DATA_SHA256 = {k: sha256_file(p) for k, p in paths.items()}
print("dataset SHA256:")
for _k, _h in DATA_SHA256.items():
    print(f"  {_k}: {_h}")

train_raw = read_json(paths["train"])
dev_raw = read_json(paths["dev"])
test_raw = read_json(paths["test"])
RAW_ROW_COUNTS = {"train": len(train_raw), "dev": len(dev_raw), "test": len(test_raw)}

train_records = dedup_labeled(train_raw, "train")
dev_unique = dedup_labeled(dev_raw, "dev")

# TEST: deliberately copy ONLY id/text. Never read/use TEST label/expert fields.
test_records = [{"id": str(r["id"]), "text": str(r["text"])} for r in test_raw]
test_ids = {r["id"] for r in test_records}
assert len(test_ids) == len(test_records), f"Duplicate TEST IDs: rows={len(test_records)} unique={len(test_ids)}"

overlap_ids = {r["id"] for r in dev_unique} & test_ids
overlap_count = len(overlap_ids)
clean_dev = [r for r in dev_unique if r["id"] not in test_ids]

if len(train_records) != EXPECTED_TRAIN:
    print("⚠️ TRAIN unique count differs from earlier audit:", len(train_records))
assert len(test_records) == EXPECTED_TEST, f"Unexpected TEST size: {len(test_records)}"
assert len(clean_dev) == EXPECTED_CLEAN_DEV, f"Unexpected CLEAN DEV size: {len(clean_dev)}"

# Accept both known server snapshots:
#   old:     DEV contains every TEST case as overlap -> excluded -> clean DEV
#   current: DEV already clean (0 overlaps)
if overlap_count == EXPECTED_TEST:
    print(f"✅ Raw DEV contains {EXPECTED_TEST} TEST-overlap cases; excluded them.")
elif overlap_count == 0 and len(dev_unique) == EXPECTED_CLEAN_DEV:
    print("✅ DEV is already leakage-clean; no TEST-overlap cases are present.")
else:
    print("⚠️ Unexpected DEV layout:",
          f"dev_unique={len(dev_unique)} overlap={overlap_count} clean_dev={len(clean_dev)}")

_train_ids = {r["id"] for r in train_records}
_clean_dev_ids = {r["id"] for r in clean_dev}
assert not (_train_ids & _clean_dev_ids), f"TRAIN↔CLEAN-DEV overlap: {len(_train_ids & _clean_dev_ids)}"
assert not (_train_ids & test_ids), f"TRAIN↔TEST overlap: {len(_train_ids & test_ids)}"
assert not (_clean_dev_ids & test_ids), f"CLEAN DEV↔TEST overlap: {len(_clean_dev_ids & test_ids)}"

# Diagnostics on (unlabelled) text statistics decide whether to lower-case.
_rng = random.Random(SEED)
ur_tr, ur_dv, ur_te = (sample_upper_ratios(rows, _rng) for rows in (train_records, clean_dev, test_records))
print(f"uppercase-letter ratio train median={np.median(ur_tr):.4f} "
      f"(>2% in {100 * np.mean(ur_tr > 0.02):.1f}% docs) | "
      f"clean-dev median={np.median(ur_dv):.4f} | test median={np.median(ur_te):.4f}")
print("newline-containing docs train: {:.1f}% | dev: {:.1f}% | test: {:.1f}%".format(
    *[100 * np.mean(["\n" in r["text"] for r in rows[:2000]])
      for rows in (train_records, clean_dev, test_records)]))

LOWERCASE = (CASE_MODE == "lower") or (CASE_MODE == "auto" and float(np.median(ur_te)) < 0.01)
print("LOWERCASE:", LOWERCASE, "| COLLAPSE_WS:", COLLAPSE_WS)

for _rows in (train_records, clean_dev, test_records):
    for _r in _rows:
        _r["text"] = normalize_text(_r["text"], LOWERCASE, COLLAPSE_WS)

del train_raw, dev_raw, test_raw, dev_unique
gc.collect()

print("TRAIN:", len(train_records), Counter(r["label"] for r in train_records))
print("CLEAN DEV:", len(clean_dev), Counter(r["label"] for r in clean_dev))
print("TEST:", len(test_records))
print("Excluded DEV↔TEST overlap:", overlap_count)
print("✅ TEST label/expert fields ignored.")
print("✅ Final CLEAN DEV contains no TEST IDs.")

Phase_4_CJPE/Train/train.json: reconstructing file:   0%|          |  0.00B /  737MB            

Phase_4_CJPE/Train/train.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Train/dev.json: reconstructing file:   0%|          |  0.00B / 45.0MB            

Phase_4_CJPE/Train/dev.json: downloading bytes:           |  0.00B            

Phase_4_CJPE/Test/test.json: reconstructing file:   0%|          |  0.00B / 35.3MB            

Phase_4_CJPE/Test/test.json: downloading bytes:           |  0.00B            

dataset SHA256:
  train: a115cc722e9de8b68ac5d41179c12d86c2d537c0de8ff16412c44c6c8d1cce72
  dev: aa2d80ae77f5d532fbc3f30e4dea2776700b8e92e71c603737d8f41a9d30b514
  test: f3498fb832e90fe7264fcb6770918eb5261409e9a952d47a2979b6ed5b153b59
train: rows=37387 unique=32305 | same-ID different-text=0 | label-conflict IDs dropped=0
dev: rows=1988 unique=994 | same-ID different-text=0 | label-conflict IDs dropped=0
✅ DEV is already leakage-clean; no TEST-overlap cases are present.
uppercase-letter ratio train median=0.0351 (>2% in 97.1% docs) | clean-dev median=0.0000 | test median=0.0000
newline-containing docs train: 0.0% | dev: 100.0% | test: 98.9%
LOWERCASE: True | COLLAPSE_WS: True
TRAIN: 32305 Counter({0: 18920, 1: 13385})
CLEAN DEV: 994 Counter({0: 497, 1: 497})
TEST: 1517
Excluded DEV↔TEST overlap: 0
✅ TEST label/expert fields ignored.
✅ Final CLEAN DEV contains no TEST IDs.


In [5]:
# CELL 5 — exact Phase-1 Model1 Prefix Encoder architecture + strict checkpoint load
class SinusoidalPositionalEncoding(nn.Module):
    """Fixed sinusoidal table (non-persistent buffer, so it is not part of the state_dict)."""
    def __init__(self, d_model: int, max_positions: int):
        super().__init__()
        pos = torch.arange(max_positions, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_positions, d_model, dtype=torch.float32)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe, persistent=False)

    def forward(self, position_ids):
        return self.pe[position_ids]


class PrefixSelfAttention(nn.Module):
    """Fused-qkv multi-head attention; masking is supplied by the caller."""
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout = dropout

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.unbind(dim=2)
        q = q.transpose(1, 2); k = k.transpose(1, 2); v = v.transpose(1, 2)

        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat([pk, k], dim=2)
            v = torch.cat([pv, v], dim=2)

        y = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=attn_mask,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=False,
        )
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.out_proj(y)
        return y, ((k, v) if use_cache else None)


class EncoderBlock(nn.Module):
    """Pre-LN transformer block: x + attn(LN(x)); x + FFN(LN(x))."""
    def __init__(self, d_model: int, n_heads: int, ffn_dim: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixSelfAttention(d_model, n_heads, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ffn_dim), nn.GELU(),
            nn.Linear(ffn_dim, d_model), nn.Dropout(dropout),
        )
        self.resid_drop = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        a, new_kv = self.attn(self.ln1(x), attn_mask=attn_mask, past_kv=past_kv, use_cache=use_cache)
        x = x + self.resid_drop(a)
        x = x + self.ff(self.ln2(x))
        return x, new_kv


class PrefixEncoderLM(nn.Module):
    """Phase-1 Model1: bidirectional prefix + causal suffix, tied input/output embedding."""
    def __init__(self, vocab_size, d_model, n_layers, n_heads, ffn_dim, dropout, max_positions):
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_positions = max_positions
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_positions)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, n_heads, ffn_dim, dropout) for _ in range(n_layers)
        ])
        self.final_ln = nn.LayerNorm(d_model)
        self.apply(self._init_weights)

        scaled_std = 0.02 / math.sqrt(2 * n_layers)
        for block in self.blocks:
            nn.init.normal_(block.attn.out_proj.weight, mean=0.0, std=scaled_std)
            nn.init.normal_(block.ff[2].weight, mean=0.0, std=scaled_std)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    @staticmethod
    def prefix_mask(T: int, prefix_len: int, device):
        """True = may attend. Prefix is fully bidirectional; the suffix is causal."""
        q = torch.arange(T, device=device)[:, None]
        k = torch.arange(T, device=device)[None, :]
        p = int(prefix_len)
        allowed = ((q < p) & (k < p)) | ((q >= p) & (k <= q))
        return allowed[None, None, :, :]

    def forward_hidden(self, input_ids, prefix_len=None, past_key_values=None, use_cache=False,
                       position_ids=None, key_valid=None):
        B, T = input_ids.shape
        past_len = 0 if past_key_values is None else past_key_values[0][0].size(2)
        if past_len + T > self.max_positions:
            raise ValueError(f"context {past_len + T} exceeds max_positions={self.max_positions}")

        if position_ids is None:
            position_ids = torch.arange(past_len, past_len + T, device=input_ids.device)[None, :].expand(B, T)

        x = self.token_emb(input_ids) + self.pos(position_ids).to(self.token_emb.weight.dtype)
        x = self.drop(x)

        if key_valid is not None:
            # Full bidirectional encoder view, but never attend to right-padding.
            attn_mask = key_valid[:, None, None, :]
        elif past_key_values is None and prefix_len is not None and int(prefix_len) < T:
            attn_mask = self.prefix_mask(T, prefix_len, input_ids.device)
        else:
            attn_mask = None

        new_cache = [] if use_cache else None
        for i, block in enumerate(self.blocks):
            past = None if past_key_values is None else past_key_values[i]
            x, kv = block(x, attn_mask=attn_mask, past_kv=past, use_cache=use_cache)
            if use_cache:
                new_cache.append(kv)
        x = self.final_ln(x)
        return x, new_cache

    def project(self, hidden):
        return F.linear(hidden, self.token_emb.weight)


def build_backbone() -> PrefixEncoderLM:
    return PrefixEncoderLM(
        vocab_size=VOCAB_SIZE, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
        ffn_dim=FFN_DIM, dropout=DROPOUT, max_positions=MAX_POSITIONS,
    )


def expected_param_count(vocab: int, d: int, layers: int, ffn: int) -> int:
    """Closed-form parameter count (the sinusoidal table is a buffer, not a parameter).
    per block = ln1(2d) + qkv(3d²+3d) + out(d²+d) + ln2(2d) + ff1(d·f+f) + ff2(f·d+d)."""
    per_layer = 4 * d * d + 2 * d * ffn + 9 * d + ffn
    return vocab * d + layers * per_layer + 2 * d


def safe_torch_load(path):
    """weights_only=True first (no arbitrary unpickling); fall back for trusted legacy files."""
    try:
        return torch.load(path, map_location="cpu", weights_only=True)
    except Exception as e:
        print(f"⚠️ weights_only load failed ({type(e).__name__}); retrying full unpickle of trusted file {path}")
        return torch.load(path, map_location="cpu", weights_only=False)


# ── load Phase-1 checkpoint ──────────────────────────────────────────────────────────────
def load_phase1_backbone(path, verbose: bool = True) -> PrefixEncoderLM:
    """Build Model1 and load the Phase-1 weights with strict=True."""
    obj = safe_torch_load(path)
    state = obj["model"] if isinstance(obj, dict) and "model" in obj else obj
    assert tuple(state["token_emb.weight"].shape) == (VOCAB_SIZE, D_MODEL), state["token_emb.weight"].shape
    backbone = build_backbone()
    backbone.load_state_dict(state, strict=True)          # raises on any missing/unexpected key
    if verbose and isinstance(obj, dict) and "config" in obj:
        print("checkpoint config:", obj["config"])
    del obj, state
    gc.collect()
    return backbone

bb = load_phase1_backbone(PHASE1_CKPT)
n_params = sum(p.numel() for p in bb.parameters())
assert n_params == expected_param_count(VOCAB_SIZE, D_MODEL, N_LAYERS, FFN_DIM), n_params
assert len(tokenizer) == VOCAB_SIZE, (len(tokenizer), VOCAB_SIZE)
print(f"Strict Model1 load PASS | params={n_params / 1e6:.3f}M (matches analytic count)")

checkpoint config: {'vocab_size': 24576, 'd_model': 768, 'n_layers': 14, 'n_heads': 12, 'ffn_dim': 3072, 'dropout': 0.0, 'max_positions': 2048, 'seq_len': 1536, 'prompt_p10_tokens': 1050, 'prompt_p95_tokens': 1139, 'expected_cont_tokens': 384}
Strict Model1 load PASS | params=118.106M (matches analytic count)


In [6]:
# CELL 6 — strict BitFit + hierarchical classifier
def enable_strict_bitfit(backbone: nn.Module) -> list:
    """Freeze every backbone parameter, then unfreeze only tensors named *.bias. Returns their names."""
    for p in backbone.parameters():
        p.requires_grad = False

    bias_names = []
    for name, p in backbone.named_parameters():
        if name.endswith(".bias"):
            p.requires_grad = True
            bias_names.append(name)

    assert bias_names, "No backbone bias parameters found."
    assert all(p.requires_grad == name.endswith(".bias") for name, p in backbone.named_parameters()), \
        "BitFit invariant failed: a non-bias backbone parameter is trainable."
    return bias_names


class HierarchicalCJPE(nn.Module):
    """Encode each of K chunks with the backbone, pool (last+mean), project, attention-pool over chunks."""
    def __init__(self, backbone: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.chunk_proj = nn.Sequential(
            nn.Linear(2 * D_MODEL, CHUNK_PROJ_DIM), nn.GELU(),
            nn.LayerNorm(CHUNK_PROJ_DIM), nn.Dropout(HEAD_DROPOUT),
        )
        self.chunk_pos = nn.Embedding(N_CHUNKS, CHUNK_PROJ_DIM)
        self.attn_score = nn.Sequential(
            nn.Linear(CHUNK_PROJ_DIM, 128), nn.Tanh(), nn.Linear(128, 1)
        )
        self.fc = nn.Sequential(
            nn.Linear(2 * CHUNK_PROJ_DIM, 256), nn.GELU(),
            nn.Dropout(HEAD_DROPOUT), nn.Linear(256, 2),
        )

    def forward(self, input_ids, valid):
        B, K, T = input_ids.shape
        x = input_ids.view(B * K, T)
        m = valid.view(B * K, T)

        # Model1 is encoder/hybrid: use fully bidirectional chunk encoding,
        # with right-padding excluded from attention.
        h, _ = self.backbone.forward_hidden(x, key_valid=m)

        mf = m.unsqueeze(-1).to(h.dtype)
        mean = (h * mf).sum(1) / mf.sum(1).clamp(min=1)

        last_idx = m.long().sum(1).clamp(min=1) - 1
        last = h[torch.arange(B * K, device=h.device), last_idx]

        z = self.chunk_proj(torch.cat([last.float(), mean.float()], dim=-1)).view(B, K, -1)
        z = z + self.chunk_pos(torch.arange(K, device=z.device)[None, :])
        w = torch.softmax(self.attn_score(z).squeeze(-1), dim=-1)

        att = (z * w.unsqueeze(-1)).sum(1)
        mx = z.max(1).values
        return self.fc(torch.cat([att, mx], dim=-1)), w


def count_trainable(model: nn.Module, backbone: bool) -> int:
    """Trainable parameters inside (backbone=True) or outside (False) the Phase-1 backbone."""
    return sum(p.numel() for n, p in model.named_parameters()
               if p.requires_grad and n.startswith("backbone.") == backbone)


BITFIT_BIAS_NAMES = enable_strict_bitfit(bb)
model = HierarchicalCJPE(bb).to(DEVICE)

bitfit_params = count_trainable(model, backbone=True)
head_params = count_trainable(model, backbone=False)
total_params = sum(p.numel() for p in model.parameters())

print("BitFit backbone bias tensors:", len(BITFIT_BIAS_NAMES))
print("first BitFit names:", BITFIT_BIAS_NAMES[:12])
print(f"BitFit trainable={bitfit_params / 1e6:.4f}M | head={head_params / 1e6:.3f}M | total model={total_params / 1e6:.3f}M")
print(f"Total trainable={(bitfit_params + head_params) / 1e6:.3f}M ({100 * (bitfit_params + head_params) / total_params:.3f}% of model)")

# Pure-BitFit guard: no token embeddings, attention/FF weights or LN gains may train.
_non_bias_trainable = [n for n, p in model.backbone.named_parameters() if p.requires_grad and not n.endswith(".bias")]
assert not _non_bias_trainable, _non_bias_trainable

BitFit backbone bias tensors: 85
first BitFit names: ['blocks.0.ln1.bias', 'blocks.0.attn.qkv.bias', 'blocks.0.attn.out_proj.bias', 'blocks.0.ln2.bias', 'blocks.0.ff.0.bias', 'blocks.0.ff.2.bias', 'blocks.1.ln1.bias', 'blocks.1.attn.qkv.bias', 'blocks.1.attn.out_proj.bias', 'blocks.1.ln2.bias', 'blocks.1.ff.0.bias', 'blocks.1.ff.2.bias']
BitFit trainable=0.1190M | head=0.839M | total model=118.945M
Total trainable=0.958M (0.806% of model)


In [7]:
# CELL 7 — tokenize/select HEAD-MIDDLE-EVIDENCE-TAIL
# Slice-based: long cases are never fully BPE-encoded (400k-word judgments would be far too slow).
SLICE_CHARS = CHUNK_LEN * 8          # ~3k characters  >>  384 BPE tokens

def _snap_forward(text: str, i: int) -> int:
    """Move index i just past the next whitespace (so a slice never starts mid-word)."""
    i = max(0, min(int(i), max(0, len(text) - 1)))
    m = re.search(r"\s", text[i:i + 400])
    return i + m.end() if m else i

def _window_start(text: str, center: int, width: int) -> int:
    return _snap_forward(text, max(0, int(center) - width // 2))

def _last_evidence_position(text: str) -> int:
    """Character index of the last disposition cue (-1 if none)."""
    low = text.lower()
    return max((low.rfind(cue) for cue in EVIDENCE_CUES), default=-1)

def _token_views(ids: list) -> list:
    """head / 50% / 75% (late, stands in for 'evidence') / tail windows of an already-tokenised text."""
    L = len(ids)
    def window(frac: float) -> list:
        center = int(round(frac * (L - 1)))
        s = max(0, min(max(0, L - CHUNK_LEN), center - CHUNK_LEN // 2))
        return ids[s:s + CHUNK_LEN]
    return [ids[:CHUNK_LEN], window(0.50), window(0.75), ids[-CHUNK_LEN:]]

def select_chunks(text: str, stats: Optional[Counter] = None):
    """Four BPE views of a case: head / middle / evidence-or-late / tail.
    Short cases are tokenised exactly; long cases are tokenised from character slices.
    Returns (list of 4 uint16 arrays, estimated full length in tokens)."""
    stats = stats if stats is not None else Counter()
    n = len(text); W = SLICE_CHARS

    if n <= 4 * W:
        stats["short_doc"] += 1
        ids = tokenizer.encode(text) or [tokenizer.unk_token_id]
        views, est = _token_views(ids), len(ids)
    else:
        stats["long_doc"] += 1
        head = tokenizer.encode(text[:W])

        # True character-centred middle slice.
        ms = _window_start(text, n * 0.50, W)
        mid = tokenizer.encode(text[ms:ms + W])

        # Evidence chunk: centred on the LAST neutral disposition cue; else a late (75%) region.
        ep = _last_evidence_position(text)
        stats["cue_found" if ep >= 0 else "cue_missing"] += 1
        es = _window_start(text, ep if ep >= 0 else int(n * 0.75), W)
        evidence = tokenizer.encode(text[es:es + W])

        ts = _snap_forward(text, max(0, n - W))
        tail = tokenizer.encode(text[ts:])

        if min(len(head), len(mid), len(evidence), len(tail)) < CHUNK_LEN:
            # Rare token-density / pathological fallback: exact full tokenisation.
            stats["fallback_full_tokenisation"] += 1
            ids = tokenizer.encode(text) or [tokenizer.unk_token_id]
            views, est = _token_views(ids), len(ids)
        else:
            views = [head[:CHUNK_LEN], mid[:CHUNK_LEN], evidence[:CHUNK_LEN], tail[-CHUNK_LEN:]]
            est = int(n * len(head) / max(W, 1))          # statistics only

    return [np.asarray(v, dtype=np.uint16) for v in views], est


class CaseDataset(Dataset):
    """Pre-tokenised cases: each item = 4 uint16 chunk arrays (+ label when labelled)."""
    def __init__(self, records: list, labeled: bool, name: str = ""):
        self.records = records; self.labeled = labeled
        self.chunks = []; self.full_lens = []; self.stats = Counter()
        t0 = time.time()
        for i, r in enumerate(records):
            ch, L = select_chunks(r["text"], self.stats)
            self.chunks.append(ch); self.full_lens.append(L)
            if (i + 1) % 5000 == 0:
                print(f"[{name}] tokenized {i + 1}/{len(records)} elapsed={time.time() - t0:.0f}s", flush=True)
        print(f"[{name}] done {len(records)} in {time.time() - t0:.0f}s | est. median BPE="
              f"{np.median(self.full_lens):.0f} p95={np.percentile(self.full_lens, 95):.0f}")
        print(f"[{name}]    chunk-selection stats:", dict(self.stats))

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int) -> dict:
        r = self.records[i]; d = {"id": r["id"], "chunks": self.chunks[i]}
        if self.labeled:
            d["label"] = int(r["label"])
        return d


def collate_cases(batch: list) -> dict:
    """Right-pad every chunk to CHUNK_LEN and build the boolean validity mask."""
    B = len(batch)
    ids = torch.full((B, N_CHUNKS, CHUNK_LEN), tokenizer.pad_token_id, dtype=torch.long)
    valid = torch.zeros((B, N_CHUNKS, CHUNK_LEN), dtype=torch.bool)
    for b, x in enumerate(batch):
        for k, c in enumerate(x["chunks"]):
            z = torch.from_numpy(c.astype(np.int64)); n = min(len(z), CHUNK_LEN)
            ids[b, k, :n] = z[:n]; valid[b, k, :n] = True
    out = {"input_ids": ids, "valid": valid, "ids": [x["id"] for x in batch]}
    if "label" in batch[0]:
        out["labels"] = torch.tensor([x["label"] for x in batch], dtype=torch.long)
    return out


def make_loader(ds, batch_size: int, *, shuffle: bool = False, seed: Optional[int] = None,
                persistent: bool = True) -> DataLoader:
    """DataLoader factory. A shuffled loader gets its own seeded generator (reproducible order)."""
    gen = None
    if shuffle:
        gen = torch.Generator(); gen.manual_seed(SEED if seed is None else seed)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, generator=gen,
                      collate_fn=collate_cases, num_workers=NUM_WORKERS,
                      pin_memory=DEVICE.type == "cuda",
                      persistent_workers=persistent and NUM_WORKERS > 0)


train_ds = CaseDataset(train_records, True, "train")
dev_ds = CaseDataset(clean_dev, True, "dev")
test_ds = CaseDataset(test_records, False, "test")

train_loader = make_loader(train_ds, MICRO_BATCH_SIZE, shuffle=True)
dev_loader = make_loader(dev_ds, EVAL_BATCH_SIZE)
test_loader = make_loader(test_ds, EVAL_BATCH_SIZE, persistent=False)
print("batches:", len(train_loader), len(dev_loader), len(test_loader))


@torch.no_grad()
def verify_padding_mask_path(n_valid: int = 100, warn_rel: float = 0.02) -> None:
    """Padding must not leak: valid-token states of a right-padded chunk (key_valid mask)
    must equal those of the same tokens run alone, un-padded."""
    was_training = model.training
    model.eval()
    b = next(iter(dev_loader))
    ids0, valid0 = b["input_ids"][0, 0], b["valid"][0, 0]
    n = min(n_valid, int(valid0.sum()), CHUNK_LEN // 2)        # guarantee real right-padding
    if n < 8:
        print("padding-mask check skipped (first dev chunk too short)")
        model.train(was_training); return

    alone = ids0[:n][None].to(DEVICE)
    padded = torch.full((1, CHUNK_LEN), tokenizer.pad_token_id, dtype=torch.long, device=DEVICE)
    padded[:, :n] = alone
    key_valid = torch.zeros((1, CHUNK_LEN), dtype=torch.bool, device=DEVICE); key_valid[:, :n] = True

    h_alone, _ = model.backbone.forward_hidden(alone)
    h_pad, _ = model.backbone.forward_hidden(padded, key_valid=key_valid)
    assert bool(torch.isfinite(h_alone).all() and torch.isfinite(h_pad[key_valid]).all()), \
        "Non-finite hidden states"
    rel = ((h_alone[0] - h_pad[0, :n]).abs().max() / h_alone.abs().max().clamp(min=1e-6)).item()
    print(f"Model1 bidirectional + padding-mask path: finite ✓ | padded-vs-alone max rel. diff={rel:.2e}")
    if rel > warn_rel:
        print("⚠️ padded and un-padded hidden states differ noticeably — check the key_valid mask path.")
    assert rel < 0.10, (
        f"Padding-mask sanity FAILED: relative difference={rel:.3e}. "
        "Valid-token states should not materially depend on appended right-padding."
    )
    model.train(was_training)


@torch.no_grad()
def phase1_lm_sanity(n_docs: int = 16) -> None:
    """Tokenizer/checkpoint sanity using Model1's own prefix-LM objective.
    Scores only the causal suffix after a bidirectional prefix, so no future-token leakage."""
    was_training = model.training
    model.backbone.eval()
    losses = []
    for i in range(min(n_docs, len(train_ds))):
        ids = torch.from_numpy(train_ds.chunks[i][1].astype(np.int64))[None].to(DEVICE)   # middle view
        if ids.size(1) < 64:
            continue
        x, y = ids[:, :-1], ids[:, 1:]
        p = max(1, x.size(1) // 2)
        with autocast():
            h, _ = model.backbone.forward_hidden(x, prefix_len=p)
            lg = model.backbone.project(h[:, p - 1:]).float()
        losses.append(F.cross_entropy(lg.reshape(-1, lg.size(-1)), y[:, p - 1:].reshape(-1)).item())
    model.train(was_training)

    if not losses:
        print("LM sanity skipped (docs too short)"); return
    m = float(np.mean(losses))
    print(f"Phase-1 Model1 prefix-LM sanity: suffix loss={m:.3f} "
          f"(ppl {math.exp(min(m, 20)):.1f}) | chance={math.log(VOCAB_SIZE):.2f}")
    if m > 7.0:
        print("⚠️ LM loss is unusually high; verify that the tokenizer belongs to Model1.")
    assert m < 8.5, (f"Phase-1 Model1 LM sanity FAILED: mean loss={m:.3f}. Most likely the wrong tokenizer "
                     "was selected. Set TOKENIZER_OVERRIDE or EXPECTED_TOKENIZER_SHA256.")

verify_padding_mask_path()
phase1_lm_sanity()

[train] tokenized 5000/32305 elapsed=15s
[train] tokenized 10000/32305 elapsed=25s
[train] tokenized 15000/32305 elapsed=38s
[train] tokenized 20000/32305 elapsed=52s
[train] tokenized 25000/32305 elapsed=63s
[train] tokenized 30000/32305 elapsed=75s
[train] done 32305 in 79s | est. median BPE=4416 p95=20119
[train]    chunk-selection stats: {'long_doc': 15940, 'cue_found': 12469, 'cue_missing': 3471, 'short_doc': 16365}
[dev] done 994 in 3s | est. median BPE=6232 p95=21137
[dev]    chunk-selection stats: {'long_doc': 657, 'cue_found': 480, 'cue_missing': 177, 'short_doc': 337}
[test] done 1517 in 4s | est. median BPE=6505 p95=22861
[test]    chunk-selection stats: {'long_doc': 1053, 'cue_found': 772, 'short_doc': 464, 'cue_missing': 281}
batches: 16153 166 253
Model1 bidirectional + padding-mask path: finite ✓ | padded-vs-alone max rel. diff=4.05e-05
Phase-1 Model1 prefix-LM sanity: suffix loss=3.821 (ppl 45.7) | chance=10.11


In [8]:
# CELL 8 — training engine, exact metric, threshold search
def compute_class_weights(records: list) -> torch.Tensor:
    """Inverse-frequency class weights, normalised to mean 1."""
    counts = Counter(r["label"] for r in records)
    w = torch.tensor([len(records) / (2 * counts[i]) for i in (0, 1)], dtype=torch.float32, device=DEVICE)
    return w / w.mean()

def make_groups(m: nn.Module) -> list:
    """Two AdamW groups: BitFit biases (no weight decay) and the new head."""
    bitfit = [p for n, p in m.named_parameters() if p.requires_grad and n.startswith("backbone.")]
    head = [p for n, p in m.named_parameters() if p.requires_grad and not n.startswith("backbone.")]
    assert bitfit and head
    return [
        {"params": bitfit, "lr": BITFIT_LR, "weight_decay": 0.0, "name": "bitfit_biases"},
        {"params": head, "lr": HEAD_LR, "weight_decay": WEIGHT_DECAY, "name": "head"},
    ]

def build_optimizer(m: nn.Module) -> torch.optim.Optimizer:
    groups = make_groups(m)
    try:
        return torch.optim.AdamW(groups, fused=(DEVICE.type == "cuda"))
    except Exception:
        return torch.optim.AdamW(groups)

def cosine_warmup_lambda(total_updates: int, warmup_ratio: Optional[float] = None):
    """Linear warm-up then cosine decay to 0, as a LambdaLR multiplier. Returns (fn, warmup_updates)."""
    warmup = max(1, int(total_updates * (WARMUP_RATIO if warmup_ratio is None else warmup_ratio)))
    def mult(step: int) -> float:
        if step < warmup:
            return max(1e-8, step / warmup)
        p = (step - warmup) / max(1, total_updates - warmup)
        return 0.5 * (1 + math.cos(math.pi * min(max(p, 0), 1)))
    return mult, warmup


class AccumulatingTrainer:
    """Gradient-accumulation engine shared by the DEV-selection run (Cell 9) and the refit (Cell 11).

    * fp16 autocast + GradScaler (init scale AMP_INIT_SCALE)
    * a non-finite micro-batch loss is skipped (never back-propagated)
    * the LR scheduler only advances on updates that were really applied
      (non-finite gradient norm or a GradScaler overflow => update counted as an overflow)
    """
    def __init__(self, model: nn.Module, class_weights: torch.Tensor, horizon_updates: int):
        self.model = model
        self.class_w = class_weights
        self.params = [p for p in model.parameters() if p.requires_grad]
        self.optimizer = build_optimizer(model)
        mult, self.warmup = cosine_warmup_lambda(horizon_updates)
        self.scheduler = torch.optim.lr_scheduler.LambdaLR(self.optimizer, mult)
        self.scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED,
                                           init_scale=AMP_INIT_SCALE, growth_interval=2000)
        self.good = 0            # finite micro-batches since the last optimizer boundary
        self.updates = 0         # optimizer steps that advanced the scheduler
        self.overflows = 0       # skipped / overflowed optimizer steps
        self.nonfinite = 0       # non-finite micro-batch losses skipped
        self.optimizer.zero_grad(set_to_none=True)

    def micro_step(self, batch: dict) -> Optional[float]:
        """Forward + backward one micro-batch. Returns the raw loss, or None if it was non-finite."""
        x = batch["input_ids"].to(DEVICE, non_blocking=True)
        v = batch["valid"].to(DEVICE, non_blocking=True)
        y = batch["labels"].to(DEVICE, non_blocking=True)
        with autocast():
            lg, _ = self.model(x, v)
            raw = F.cross_entropy(lg.float(), y, weight=self.class_w, label_smoothing=LABEL_SMOOTHING)
            loss = raw / GRAD_ACCUM_STEPS
        raw_val = float(raw.detach())
        if math.isfinite(raw_val):
            self.scaler.scale(loss).backward()
            self.good += 1
            return raw_val
        self.nonfinite += 1
        print(f"⚠️ non-finite loss; micro-batch skipped (total {self.nonfinite})")
        return None

    def maybe_update(self, boundary: bool) -> bool:
        """At an accumulation boundary: unscale, clip, step. Returns True if the step was applied."""
        if not boundary:
            return False
        applied = False
        if self.good == 0:
            self.optimizer.zero_grad(set_to_none=True)
        else:
            self.scaler.unscale_(self.optimizer)
            gn = torch.nn.utils.clip_grad_norm_(self.params, GRAD_CLIP)
            finite = bool(torch.isfinite(torch.as_tensor(gn)).item())
            old = float(self.scaler.get_scale())

            if not finite:
                # Never apply a non-finite update. This matters especially if AMP is disabled.
                self.optimizer.zero_grad(set_to_none=True)
                if AMP_ENABLED:
                    self.scaler.update(new_scale=max(old * 0.5, 1.0))
                self.overflows += 1
            else:
                # GradScaler itself still guards fp16 overflow between unscale_ and step().
                self.scaler.step(self.optimizer)
                self.scaler.update()
                new = float(self.scaler.get_scale())
                self.optimizer.zero_grad(set_to_none=True)
                if new >= old:
                    self.scheduler.step(); self.updates += 1; applied = True
                else:
                    self.overflows += 1
        self.good = 0
        return applied

    @property
    def lrs(self) -> tuple:
        return tuple(g["lr"] for g in self.optimizer.param_groups)


# ───────────────────────── metrics / threshold search ─────────────────────────
def macro_f1(y, p) -> float:
    return float(f1_score(y, p, average="macro", labels=[0, 1], zero_division=0))

def macro_f1_grid(y, prob, thresholds=THRESHOLDS) -> np.ndarray:
    """Macro-F1 for every threshold at once (vectorised; equals sklearn macro-F1, zero_division=0)."""
    y = np.asarray(y).astype(bool)
    pos = np.asarray(prob, dtype=np.float64)[None, :] >= np.asarray(thresholds, dtype=np.float64)[:, None]
    tp1 = (pos & y).sum(1); fp1 = (pos & ~y).sum(1); fn1 = (~pos & y).sum(1); tp0 = (~pos & ~y).sum(1)
    def f1(tp, fp, fn):
        den = 2 * tp + fp + fn
        return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    return 0.5 * (f1(tp1, fp1, fn1) + f1(tp0, fn1, fp1))      # class 0: fp0=fn1, fn0=fp1

def search_threshold(y, prob, smooth: int = THRESH_SMOOTH):
    """Pick the threshold whose moving-average (edge-padded) macro-F1 is highest.
    Returns (macro-F1 at that threshold, threshold, hard predictions, best raw grid F1)."""
    fs = macro_f1_grid(y, prob)
    k = max(1, int(smooth))
    if k > 1:
        if k % 2 == 0:
            k += 1
        pad = k // 2
        sm = np.convolve(np.pad(fs, (pad, pad), mode="edge"), np.ones(k) / k, mode="valid")
    else:
        sm = fs
    i = int(np.argmax(sm))
    t = float(THRESHOLDS[i])
    return float(fs[i]), t, (np.asarray(prob, dtype=np.float64) >= t).astype(np.int64), float(fs.max())


@torch.no_grad()
def predict_logits(m: nn.Module, loader: DataLoader):
    """Returns (logits [N,2], labels or None, ids, chunk-attention [N,K])."""
    m.eval()
    lgs, ys, ids, ats = [], [], [], []
    for b in loader:
        x = b["input_ids"].to(DEVICE, non_blocking=True)
        v = b["valid"].to(DEVICE, non_blocking=True)
        with autocast():
            lg, a = m(x, v)
        lgs.append(lg.float().cpu().numpy())
        ats.append(a.float().cpu().numpy())
        ids.extend(b["ids"])
        if "labels" in b:
            ys.append(b["labels"].numpy())
    return np.concatenate(lgs), (np.concatenate(ys) if ys else None), ids, np.concatenate(ats)

def _p1(lg: np.ndarray) -> np.ndarray:
    """P(class 1) from 2-class logits (numerically stable softmax)."""
    lg = lg.astype(np.float64)
    lg = lg - lg.max(-1, keepdims=True)
    e = np.exp(lg)
    return (e[:, 1] / e.sum(-1)).astype(np.float32)

def _logit(p) -> np.ndarray:
    p = np.clip(np.asarray(p, dtype=np.float64), 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))

def predict_probs(m: nn.Module, loader: DataLoader):
    lg, y, ids, att = predict_logits(m, loader)
    return _p1(lg), y, ids, att


class_w = compute_class_weights(train_records)
print("class weights:", class_w.tolist())

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM_STEPS)
total_updates = NUM_EPOCHS * updates_per_epoch
trainer = AccumulatingTrainer(model, class_w, total_updates)

print("updates/epoch:", updates_per_epoch, "| total updates:", total_updates, "| warm-up updates:", trainer.warmup)
print("trainable backbone bias params:", count_trainable(model, backbone=True))

class weights: [0.8286643028259277, 1.1713356971740723]
updates/epoch: 2020 | total updates: 4040 | warm-up updates: 242
trainable backbone bias params: 119040


In [9]:
# CELL 9 — BitFit training + CLEAN DEV selection
BEST = ROOT / "best_bitfit_cjpe.pt"
BEST_TMP = BEST.with_suffix(".pt.tmp")
BEST_TMP.unlink(missing_ok=True)

def trainable_state_dict(m: nn.Module) -> dict:
    """CPU copy of the trainable tensors only (BitFit biases + head); the frozen backbone is re-loaded from Phase 1."""
    names = {n for n, p in m.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in m.state_dict().items() if k in names}

def load_trainable_state(m: nn.Module, state: dict) -> None:
    expected = {n for n, p in m.named_parameters() if p.requires_grad}
    assert set(state) == expected, (
        f"Trainable-state mismatch: missing={sorted(expected - set(state))[:5]} "
        f"extra={sorted(set(state) - expected)[:5]}")
    missing, unexpected = m.load_state_dict(state, strict=False)
    assert not unexpected, unexpected
    # Missing *frozen* tensors are expected; every trainable tensor must have loaded.
    assert not (expected & set(missing)), sorted(expected & set(missing))[:10]

def compute_eval_steps(n_batches: int, per_epoch: int, accum: int) -> list:
    """Micro-batch indices at which DEV is evaluated: aligned to accumulation boundaries."""
    steps = []
    for j in range(1, per_epoch + 1):
        s = int(round(n_batches * j / per_epoch))
        s = max(accum, (s // accum) * accum) if s < n_batches else n_batches
        steps.append(s)
    return sorted(set(steps))

history: list = []
best_f1, best_thr, best_frac, bad_evals = -1.0, 0.5, None, 0
n_batches = len(train_loader)
eval_steps = set(compute_eval_steps(n_batches, EVAL_PER_EPOCH, GRAD_ACCUM_STEPS))
print("DEV eval batches:", sorted(eval_steps))

stop = False
for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    run, nr, ep0 = 0.0, 0, time.time()

    for step, batch in enumerate(train_loader, 1):
        loss_val = trainer.micro_step(batch)
        if loss_val is not None:
            run += loss_val; nr += 1
        trainer.maybe_update(step % GRAD_ACCUM_STEPS == 0 or step == n_batches)

        if step % 500 == 0 or step == n_batches:
            elapsed = time.time() - ep0
            eta = (elapsed / max(step, 1)) * (n_batches - step)
            lr_bias, lr_head = trainer.lrs
            print(f"epoch={epoch} batch={step}/{n_batches} loss={run / max(nr, 1):.4f} "
                  f"elapsed={elapsed / 60:.1f}m ETA_epoch={eta / 60:.1f}m "
                  f"lr_bias={lr_bias:.2e} lr_head={lr_head:.2e}", flush=True)

        if step in eval_steps:
            prob, ydev, _, att = predict_probs(model, dev_loader)
            f, t, _, grid_best = search_threshold(ydev, prob)
            row = {
                "epoch": epoch,
                "epoch_frac": (epoch - 1) + step / n_batches,
                "batch": step,
                "macro_f1": f,
                "threshold": t,
                "grid_best_macro_f1": grid_best,
                "f1_at_0.5": macro_f1(ydev, (prob >= .5).astype(np.int64)),
                "train_loss": run / max(nr, 1),
                "global_update": trainer.updates,
                "amp_overflows": trainer.overflows,
                "nonfinite_skipped": trainer.nonfinite,
                "attention_mean": att.mean(0).tolist(),
            }
            history.append(row)
            save_json(history, ROOT / "history.json", indent=2)
            print("DEV:", row)

            if f > best_f1 + 1e-12:
                best_f1, best_thr, best_frac, bad_evals = f, t, row["epoch_frac"], 0
                torch.save({
                    "trainable_state": trainable_state_dict(model),
                    "best_macro_f1": float(best_f1),
                    "best_threshold": float(best_thr),
                    "best_epoch_frac": float(best_frac),
                    "bitfit_bias_names": list(BITFIT_BIAS_NAMES),
                    "bitfit_lr": BITFIT_LR,
                    "head_lr": HEAD_LR,
                    "seed": SEED,
                    "method": "strict BitFit",
                }, BEST_TMP)
                os.replace(BEST_TMP, BEST)
                print(f"✅ NEW BEST Macro-F1={best_f1:.6f} threshold={best_thr:.3f}")
            else:
                bad_evals += 1

            if bad_evals >= PATIENCE_EVALS:
                print("Early stopping.")
                stop = True
                break
            model.train()

    if stop:
        break

assert BEST.is_file(), "no DEV evaluation produced a checkpoint"
print("DONE best F1/thr/frac:", best_f1, best_thr, best_frac)

DEV eval batches: [5384, 10768, 16153]
epoch=1 batch=500/16153 loss=0.7077 elapsed=0.9m ETA_epoch=29.7m lr_bias=2.05e-04 lr_head=1.28e-04
epoch=1 batch=1000/16153 loss=0.7117 elapsed=1.9m ETA_epoch=29.3m lr_bias=4.13e-04 lr_head=2.58e-04
epoch=1 batch=1500/16153 loss=0.7180 elapsed=2.9m ETA_epoch=28.4m lr_bias=6.18e-04 lr_head=3.86e-04
epoch=1 batch=2000/16153 loss=0.7169 elapsed=3.9m ETA_epoch=27.5m lr_bias=8.00e-04 lr_head=5.00e-04
epoch=1 batch=2500/16153 loss=0.7161 elapsed=4.9m ETA_epoch=26.5m lr_bias=7.99e-04 lr_head=5.00e-04
epoch=1 batch=3000/16153 loss=0.7140 elapsed=5.8m ETA_epoch=25.6m lr_bias=7.98e-04 lr_head=4.98e-04
epoch=1 batch=3500/16153 loss=0.7124 elapsed=6.8m ETA_epoch=24.6m lr_bias=7.95e-04 lr_head=4.97e-04
epoch=1 batch=4000/16153 loss=0.7093 elapsed=7.8m ETA_epoch=23.6m lr_bias=7.91e-04 lr_head=4.94e-04
epoch=1 batch=4500/16153 loss=0.7073 elapsed=8.7m ETA_epoch=22.6m lr_bias=7.86e-04 lr_head=4.91e-04
epoch=1 batch=5000/16153 loss=0.7063 elapsed=9.7m ETA_epoch=21

In [10]:
# CELL 10 — exact supplied Phase-4 evaluator on CLEAN DEV
best_obj = safe_torch_load(BEST)
load_trainable_state(model, best_obj["trainable_state"])
best_thr = float(best_obj["best_threshold"])

dev_prob, dev_y, dev_ids, dev_att = predict_probs(model, dev_loader)
dev_pred = (dev_prob.astype(np.float64) >= best_thr).astype(np.int64)

truth = [{"id": str(i), "gold": int(y)} for i, y in zip(dev_ids, dev_y)]
pred = [{"id": str(i), "pred": int(p)} for i, p in zip(dev_ids, dev_pred)]

# Verbatim copy of the organisers' evaluate script — do not edit.
EVALUATOR_CODE = r'''#!/usr/bin/env python3
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score
def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out
def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out
def main():
    ap=argparse.ArgumentParser()
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True)
    a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e:
        print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''

def run_official_evaluator(truth_rows: list, pred_rows: list) -> dict:
    """Run the organisers' script in a subprocess; returns the parsed metrics."""
    with tempfile.TemporaryDirectory() as td:
        tp, pp, ep = Path(td) / "truth.json", Path(td) / "prediction.json", Path(td) / "evaluate_phase4.py"
        save_json(truth_rows, tp); save_json(pred_rows, pp)
        ep.write_text(EVALUATOR_CODE, encoding="utf-8")
        res = subprocess.run([sys.executable, str(ep), "--truth", str(tp), "--prediction", str(pp)],
                             capture_output=True, text=True)
    print(res.stdout)
    if res.stderr:
        print(res.stderr)
    assert res.returncode == 0, "official evaluator failed"
    return {k.strip(): float(v) for k, v in re.findall(r"^(Macro-F1|Precision|Recall):\s*([0-9.]+)", res.stdout, re.M)}

eval_metrics = run_official_evaluator(truth, pred)
print("✅ Exact Phase-4 evaluator PASS")

# The reloaded best checkpoint must reproduce the F1 recorded during training (tiny fp16 jitter is fine).
_drift = abs(eval_metrics["Macro-F1"] - float(best_obj["best_macro_f1"]))
print(f"evaluator Macro-F1 vs training-time DEV F1: |Δ|={_drift:.5f}")
if _drift > 5e-3:
    print("⚠️ Reloaded checkpoint does not reproduce the recorded DEV F1 — inspect the save/load path.")

np.savez_compressed(
    ROOT / "dev_probs.npz",
    prob=dev_prob.astype(np.float32),
    y=dev_y.astype(np.int64),
    id=np.asarray(dev_ids, dtype=str),            # loads without allow_pickle
    threshold=np.asarray([best_thr], dtype=np.float32),
    chunk_attention=dev_att.astype(np.float32),
)

Examples: 994
Macro-F1: 0.603042
Precision: 0.612200
Recall: 0.565392

✅ Exact Phase-4 evaluator PASS
evaluator Macro-F1 vs training-time DEV F1: |Δ|=0.00000


In [11]:
# CELL 11 — fresh Model1+BitFit and clean TRAIN+DEV refit
REFIT_SEED = SEED + 1

def fresh_model() -> nn.Module:
    """New Phase-1 backbone (strict load) with strict BitFit + a newly initialised head."""
    backbone = load_phase1_backbone(PHASE1_CKPT, verbose=False)
    enable_strict_bitfit(backbone)
    return HierarchicalCJPE(backbone).to(DEVICE)

def run_refit(best: dict):
    """Re-train from scratch on TRAIN + clean DEV for the number of micro-batches that was optimal on DEV.
    Returns (refit model, info dict)."""
    all_records = train_records + clean_dev
    seed_everything(REFIT_SEED)
    all_loader = make_loader(ConcatDataset([train_ds, dev_ds]), MICRO_BATCH_SIZE,
                             shuffle=True, seed=REFIT_SEED)
    m = fresh_model()

    n_all = len(all_loader)
    target_batches = max(
        GRAD_ACCUM_STEPS,
        int(round(float(best["best_epoch_frac"]) * n_all / GRAD_ACCUM_STEPS)) * GRAD_ACCUM_STEPS,
    )
    upe_all = math.ceil(n_all / GRAD_ACCUM_STEPS)
    horizon_updates = NUM_EPOCHS * upe_all if REFIT_MATCH_ORIGINAL_SCHEDULE \
        else math.ceil(target_batches / GRAD_ACCUM_STEPS)
    r = AccumulatingTrainer(m, compute_class_weights(all_records), horizon_updates)
    print("refit target micro-batches:", target_batches, "| LR horizon updates:", horizon_updates)

    seen, t0 = 0, time.time()
    while seen < target_batches:
        m.train()
        for batch in all_loader:
            if seen >= target_batches:
                break
            r.micro_step(batch)
            seen += 1
            r.maybe_update(seen % GRAD_ACCUM_STEPS == 0 or seen >= target_batches)
            if seen % 500 == 0 or seen >= target_batches:
                rate = (time.time() - t0) / max(seen, 1)
                print(f"refit {seen}/{target_batches} ETA={(target_batches - seen) * rate / 60:.1f}m "
                      f"overflow={r.overflows}", flush=True)

    final = ROOT / "final_refit_bitfit_cjpe.pt"
    tmp = final.with_suffix(".pt.tmp"); tmp.unlink(missing_ok=True)
    torch.save({
        "trainable_state": trainable_state_dict(m),
        "source_best_epoch_frac": float(best["best_epoch_frac"]),
        "bitfit_bias_names": list(BITFIT_BIAS_NAMES),
        "bitfit_lr": BITFIT_LR,
        "head_lr": HEAD_LR,
        "refit_seed": REFIT_SEED,
        "refit_target_batches": int(target_batches),
        "overflows": r.overflows,
        "nonfinite_skipped": r.nonfinite,
        "method": "strict BitFit",
    }, tmp)
    os.replace(tmp, final)
    info = {"target_batches": int(target_batches), "overflows": r.overflows, "nonfinite_skipped": r.nonfinite}
    del r, all_loader                       # release optimizer state / loader workers
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    print("✅ refit complete")
    return m, info

refit_model, target_batches = None, None
if RUN_REFIT:
    refit_model, refit_info = run_refit(best_obj)
    target_batches = refit_info["target_batches"]

refit target micro-batches: 33296 | LR horizon updates: 4164
refit 500/33296 ETA=64.4m overflow=0
refit 1000/33296 ETA=63.3m overflow=0
refit 1500/33296 ETA=62.3m overflow=0
refit 2000/33296 ETA=61.3m overflow=0
refit 2500/33296 ETA=60.2m overflow=0
refit 3000/33296 ETA=59.2m overflow=0
refit 3500/33296 ETA=58.3m overflow=0
refit 4000/33296 ETA=57.3m overflow=0
refit 4500/33296 ETA=56.3m overflow=0
refit 5000/33296 ETA=55.3m overflow=0
refit 5500/33296 ETA=54.3m overflow=0
refit 6000/33296 ETA=53.3m overflow=0
refit 6500/33296 ETA=52.3m overflow=0
refit 7000/33296 ETA=51.3m overflow=0
refit 7500/33296 ETA=50.3m overflow=0
refit 8000/33296 ETA=49.4m overflow=0
refit 8500/33296 ETA=48.4m overflow=0
refit 9000/33296 ETA=47.4m overflow=0
refit 9500/33296 ETA=46.5m overflow=0
refit 10000/33296 ETA=45.5m overflow=0
refit 10500/33296 ETA=44.5m overflow=0
refit 11000/33296 ETA=43.5m overflow=0
refit 11500/33296 ETA=42.6m overflow=0
refit 12000/33296 ETA=41.6m overflow=0
refit 12500/33296 ETA=4

In [12]:
# CELL 12 — TEST inference + label-free calibration
held_lg, _, held_ids, held_att = predict_logits(model, test_loader)
assert held_ids == [r["id"] for r in test_records] and len(held_ids) == EXPECTED_TEST
held_prob = _p1(held_lg)

refit_prob = refit_thr = test_refit_agreement = None
if refit_model is not None:
    refit_lg, _, refit_ids, _ = predict_logits(refit_model, test_loader)
    assert refit_ids == held_ids
    refit_prob = _p1(refit_lg)
    # Label-free calibration: give the refit model the same positive rate on TEST as the DEV-selected model.
    held_rate = float((held_prob >= best_thr).mean())
    refit_thr = float(np.clip(np.quantile(refit_prob, 1 - held_rate), .05, .95))
    test_refit_agreement = float(((held_prob >= best_thr) == (refit_prob >= refit_thr)).mean())
    print(f"heldout positive rate={held_rate:.3f} | refit thr={refit_thr:.3f} | agreement={test_refit_agreement:.4f}")

np.savez_compressed(
    ROOT / "test_probs.npz",
    heldout=held_prob.astype(np.float32),
    refit=(refit_prob.astype(np.float32) if refit_prob is not None else np.array([], dtype=np.float32)),
    id=np.asarray(held_ids, dtype=str),
    heldout_threshold=np.asarray([best_thr], dtype=np.float32),
    refit_threshold=np.asarray([refit_thr if refit_thr is not None else np.nan], dtype=np.float32),
)

heldout positive rate=0.467 | refit thr=0.434 | agreement=0.8167


In [13]:
# CELL 13 — three high-value Codabench ZIPs
official_ids = [r["id"] for r in test_records]
assert len(official_ids) == len(set(official_ids)) == EXPECTED_TEST

def write_zip(filename: str, labels) -> np.ndarray:
    """Write <filename> containing prediction.json, then re-open and validate it."""
    labels = np.asarray(labels, dtype=np.int64)
    rec = [{"id": str(cid), "pred": int(y)} for cid, y in zip(official_ids, labels)]
    assert len(rec) == EXPECTED_TEST and len({r["id"] for r in rec}) == EXPECTED_TEST
    assert all(type(r["pred"]) is int and r["pred"] in (0, 1) for r in rec)
    zp = ROOT / filename
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", json.dumps(rec))
    with zipfile.ZipFile(zp, "r") as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
        inside = json.loads(z.read("prediction.json").decode("utf-8"))
        assert len(inside) == EXPECTED_TEST
        assert {r["id"] for r in inside} == set(official_ids)
    print(f"✅ {filename} | positive={labels.mean():.4f} | {Counter(labels.tolist())}")
    return labels

def calibrated_margin(prob, thr) -> np.ndarray:
    """Logit distance from the decision threshold (>=0  <=>  predicted positive)."""
    return _logit(prob) - _logit(thr)

def disagreement(a, b) -> Optional[float]:
    return None if (a is None or b is None) else float(np.mean(a != b))

held_margin = calibrated_margin(held_prob, best_thr)
if refit_prob is not None:
    refit_margin = calibrated_margin(refit_prob, refit_thr)
    y_best = write_zip("submission_best.zip", ((0.5 * held_margin + 0.5 * refit_margin) >= 0).astype(np.int64))
else:
    y_best = write_zip("submission_best.zip", (held_prob >= best_thr).astype(np.int64))

y_held = write_zip("submission_heldout.zip", (held_prob >= best_thr).astype(np.int64))

y_hedge = None
if refit_prob is not None:
    y_hedge = write_zip("submission_margin_h70_r30.zip",
                        ((0.70 * held_margin + 0.30 * refit_margin) >= 0).astype(np.int64))

disagree_best_held = disagreement(y_best, y_held)
disagree_best_hedge = disagreement(y_best, y_hedge)
disagree_held_hedge = disagreement(y_held, y_hedge)
print("best vs heldout disagreement:", disagree_best_held)
if y_hedge is not None:
    print("best vs 70/30 hedge disagreement:", disagree_best_hedge)
    print("heldout vs 70/30 hedge disagreement:", disagree_held_hedge)

✅ submission_best.zip | positive=0.4661 | Counter({0: 810, 1: 707})
✅ submission_heldout.zip | positive=0.4674 | Counter({0: 808, 1: 709})
✅ submission_margin_h70_r30.zip | positive=0.4614 | Counter({0: 817, 1: 700})
best vs heldout disagreement: 0.10810810810810811
best vs 70/30 hedge disagreement: 0.03625576796308504
heldout vs 70/30 hedge disagreement: 0.07185234014502308


In [14]:
# CELL 14 — manifest
manifest = {
    "notebook_version": NOTEBOOK_VERSION,
    "phase": 4,
    "experiment": 4,
    "method": "Strict BitFit (backbone bias-only adaptation)",
    "phase1_model": "Model1 Prefix Encoder / hybrid LM",
    "checkpoint": str(PHASE1_CKPT),
    "checkpoint_sha256": sha256_file(PHASE1_CKPT),
    "tokenizer_sha256": sha256_file(TOKENIZER_FILE),
    "dataset_sha256": DATA_SHA256,
    "raw_dataset_rows": RAW_ROW_COUNTS,

    "architecture": {
        "vocab_size": VOCAB_SIZE, "d_model": D_MODEL, "n_layers": N_LAYERS,
        "n_heads": N_HEADS, "ffn_dim": FFN_DIM, "max_positions": MAX_POSITIONS,
    },

    "train_unique": len(train_records),
    "clean_dev": len(clean_dev),
    "test": len(test_records),
    "excluded_dev_test_overlap": overlap_count,

    "chunks": CHUNK_NAMES,
    "chunk_len": CHUNK_LEN,

    "bitfit_lr": BITFIT_LR,
    "head_lr": HEAD_LR,
    "bitfit_bias_tensor_count": len(BITFIT_BIAS_NAMES),
    "bitfit_bias_names": list(BITFIT_BIAS_NAMES),

    "best_dev_macro_f1": float(best_obj["best_macro_f1"]),
    "best_threshold": float(best_obj["best_threshold"]),
    "best_epoch_frac": float(best_obj["best_epoch_frac"]),
    "official_evaluator_dev": eval_metrics,

    "lowercase": bool(LOWERCASE),
    "collapse_ws": bool(COLLAPSE_WS),

    "refit_run": refit_model is not None,
    "refit_threshold": None if refit_thr is None else float(refit_thr),
    "test_refit_agreement": test_refit_agreement,
    "refit_target_batches": None if target_batches is None else int(target_batches),

    "test_positive_rate_best": float(np.mean(y_best)),

    "submission_strategies": {
        "submission_best.zip": "50/50 calibrated heldout+refit margin ensemble",
        "submission_heldout.zip": "clean-DEV-selected heldout checkpoint",
        "submission_margin_h70_r30.zip": "70/30 calibrated heldout/refit margin hedge",
    },
    "submission_disagreement": {
        "best_vs_heldout": disagree_best_held,
        "best_vs_h70_r30": disagree_best_hedge,
        "heldout_vs_h70_r30": disagree_held_hedge,
    },

    "config": config_snapshot(),
    "python_version": sys.version.split()[0],
    "torch_version": torch.__version__,
    "device": str(DEVICE),
    "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "seed": SEED,
    "test_labels_used": False,
    "overlap_dev_labels_used": False,
}
save_json(manifest, ROOT / "manifest.json", indent=2)
print(json.dumps(manifest, indent=2))

print("\nESSENTIAL OUTPUTS:")
for fn in ["best_bitfit_cjpe.pt", "final_refit_bitfit_cjpe.pt", "dev_probs.npz", "test_probs.npz",
           "history.json", "manifest.json", "submission_best.zip", "submission_heldout.zip",
           "submission_margin_h70_r30.zip"]:
    print(("✅" if (ROOT / fn).exists() else "—"), fn)

{
  "notebook_version": "v3_REVIEWED",
  "phase": 4,
  "experiment": 4,
  "method": "Strict BitFit (backbone bias-only adaptation)",
  "phase1_model": "Model1 Prefix Encoder / hybrid LM",
  "checkpoint": "/kaggle/input/datasets/aaryant23/model1-ckpts/enc_full_best (9).pt",
  "checkpoint_sha256": "017c57f393a55c96a9a24d0f98fc62775f3f4d0459b896eefe7a30437accacfc",
  "tokenizer_sha256": "9cfbe6f0e49067f63ba71fe1013021c5de9269a2b4bc6c9ae30a72e42102106d",
  "dataset_sha256": {
    "train": "a115cc722e9de8b68ac5d41179c12d86c2d537c0de8ff16412c44c6c8d1cce72",
    "dev": "aa2d80ae77f5d532fbc3f30e4dea2776700b8e92e71c603737d8f41a9d30b514",
    "test": "f3498fb832e90fe7264fcb6770918eb5261409e9a952d47a2979b6ed5b153b59"
  },
  "raw_dataset_rows": {
    "train": 37387,
    "dev": 1988,
    "test": 1517
  },
  "architecture": {
    "vocab_size": 24576,
    "d_model": 768,
    "n_layers": 14,
    "n_heads": 12,
    "ffn_dim": 3072,
    "max_positions": 2048
  },
  "train_unique": 32305,
  "clean_dev": 

## Keep / submit

Primary Codabench candidate: **`submission_best.zip`**.

Two useful hedges are also produced:

- `submission_heldout.zip` — pure clean-DEV-selected Model1+BitFit checkpoint.
- `submission_margin_h70_r30.zip` — gives more weight to the directly DEV-validated heldout model.

Keep `best_bitfit_cjpe.pt`, `final_refit_bitfit_cjpe.pt`, `dev_probs.npz`, `test_probs.npz`, `history.json`, and `manifest.json` for the final Phase-4 comparison / ensemble.